# ADNI PHC 구조 MRI 기반 Brain Age Gap (BAG): MVP 분석

이 노트북은 **ADSP PHC ComBat 보정 FreeSurfer ROI**로 정상 인지군(CN)의 뇌 나이 모델을 학습하고, 보정된 Brain Age Gap(BAG)이 CN·MCI·Dementia 집단에서 다른지 평가합니다.

**연구 질문**: 정상군의 구조 MRI로 학습한 뇌 나이 모델에서, MCI와 Dementia는 같은 실제 나이의 CN보다 더 높은 BAG를 보이는가?

이 분석은 연구·교육 목적의 집단 수준 관찰 분석입니다. 개인 치매 진단, 미래 위험 판정, 인과관계, 개인의 노화 속도를 주장하지 않습니다.

## 전체 흐름

```text
데이터 로드 → index MRI 코호트 구축 → 전처리/EDA → CN-only 모델 학습
→ validation 나이 편향 보정 → 독립 CN test 평가 → 전체 코호트 BAG 계산
→ 진단군 ANCOVA → ROI 중요도와 생물학적 의미 정리
```

왜 이 순서인가요? MRI·진단·인구통계 데이터가 정확히 결합되어야 모델 결과가 신뢰할 수 있고, 학습에 쓰지 않은 CN test가 있어야 성능이 과장되지 않습니다. MCI와 Dementia는 정상 노화 기준을 배우는 데 사용하지 않고, 마지막 BAG 비교에서만 사용합니다.

## 0. 라이브러리와 분석 원칙

Colab에서는 아래 셀을 한 번 실행합니다. `pyarrow`는 사용하지 않으며, 모든 중간 결과는 CSV로 저장합니다.

- `RID`: ADNI 참가자 식별자. 같은 RID가 train/test 양쪽에 들어가면 데이터 누수가 생깁니다.
- `ComBat`: MRI 장비·센터 차이를 줄인 사전 조화 ROI입니다.
- `N_JOBS=1`: 환경별 병렬 처리 권한 오류를 피하기 위한 안전 설정입니다.

In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib seaborn statsmodels scipy

from pathlib import Path
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import statsmodels.formula.api as smf
from scipy.stats import pearsonr
from statsmodels.stats.anova import anova_lm

warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 20260819
MAX_ALIGNMENT_DAYS = 180
MIN_FEATURE_COMPLETENESS = 0.70
N_JOBS = 1
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')

## 1. 경로와 데이터 로드

VS Code에서 프로젝트의 최상위 폴더(`brain-age-gap-adni`)를 연 뒤 이 노트북을 실행하세요. 경로 설정 셀은 현재 작업 폴더와 상위 폴더를 확인해 `data/adni_all`이 있는 프로젝트 루트를 자동으로 찾습니다. ADNI 원본 자료는 데이터 사용 계약에 따라 Git에 올리지 않습니다.

이 MVP에서 쓰는 핵심 파일은 다음 네 가지입니다.

- `ADSP_PHC_T1_FS`: ComBat 보정 구조 MRI ROI와 PHC 나이·성별·진단
- `PTDEMOG`: 교육연수 보강
- `DXSUM`: PHC 진단 결측 시 가까운 임상 진단 보강
- `ADSP_PHC_T1_FS_DATADIC`: PHC ROI의 해부학적 이름과 단위

In [ ]:
def locate_project_root():
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'adni_all').exists():
            return candidate
    raise FileNotFoundError(
        '프로젝트 루트를 찾지 못했습니다. VS Code에서 brain-age-gap-adni 폴더를 연 뒤 실행하세요.'
    )

PROJECT_ROOT = locate_project_root()
print(f'Project root: {PROJECT_ROOT}')
DATA_DIR = PROJECT_ROOT / 'data'
ADNI_ROOT = DATA_DIR / 'adni_all'
assert ADNI_ROOT.exists(), f'ADNI 원본 폴더를 찾지 못했습니다: {ADNI_ROOT}'

# 분석 정의를 바꾸면 RUN_VERSION만 올립니다. 같은 정의를 재실행할 때는 그대로 둡니다.
ANALYSIS_ID = '01_mri_bag_mvp'
RUN_VERSION = '20260820_v1'
RUN_ID = f'{ANALYSIS_ID}_{RUN_VERSION}'

PROCESSED_DIR = DATA_DIR / 'processed'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures' / RUN_ID
TABLE_DIR = PROJECT_ROOT / 'results' / 'tables' / RUN_ID
for folder in [PROCESSED_DIR, FIG_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print(f'Result run ID: {RUN_ID}')

def one_file(pattern, required=True):
    matches = sorted(ADNI_ROOT.glob(pattern))
    if required:
        assert matches, f'파일을 찾지 못했습니다: {pattern}'
    return matches[-1] if matches else None

paths = {
    # Date must follow T1_FS_ immediately; this excludes the separate *_DATADIC file.
    'phc_fs': one_file('ADSP_PHC/ADSP_PHC_T1_FS_22Jan*.csv'),
    'phc_dict': one_file('ADSP_PHC/ADSP_PHC_T1_FS_DATADIC_*.csv'),
    'demo': one_file('Subject_Characteristics/PTDEMOG_*.csv'),
    'dx': one_file('Assessments/DXSUM_*.csv'),
}

phc = pd.read_csv(paths['phc_fs'], low_memory=False)
phc_dict = pd.read_csv(paths['phc_dict'], low_memory=False)
demo = pd.read_csv(paths['demo'], low_memory=False)
dx = pd.read_csv(paths['dx'], low_memory=False)

print('PHC T1 FreeSurfer:', phc.shape)
print('PHC data dictionary:', phc_dict.shape)
print('PTDEMOG:', demo.shape)
print('DXSUM:', dx.shape)

## 2. 데이터 정렬과 index MRI 코호트

ADNI는 한 사람이 여러 방문과 여러 MRI를 가질 수 있습니다. 반복 MRI가 학습과 test에 동시에 들어가면 모델이 같은 사람을 기억할 수 있으므로, MVP에서는 **RID별 가장 이른 유효 T1 MRI 한 건**만 선택합니다. 이를 `index MRI`라고 부릅니다.

PHC의 `PHC_Age_T1`은 MRI 촬영 시점의 나이이므로 출생연도로 다시 계산한 나이보다 우선 사용합니다. 진단은 PHC 값을 우선 쓰고, 결측일 때만 MRI 날짜와 ±180일 이내 DXSUM 진단으로 보강합니다.

In [ ]:
def standardize_rid(df):
    out = df.copy()
    out['RID'] = pd.to_numeric(out['RID'], errors='coerce').astype('Int64')
    return out

def to_datetime_col(df, col):
    out = df.copy()
    out[col] = pd.to_datetime(out[col], errors='coerce')
    return out

def diagnosis_label(value):
    value = pd.to_numeric(value, errors='coerce')
    return {1: 'CN', 2: 'MCI', 3: 'Dementia'}.get(value, np.nan)

def nearest_by_rid(left, right, left_date, right_date, fields, max_days=MAX_ALIGNMENT_DAYS):
    """For each left row, attach fields from the nearest dated right row with the same RID."""
    base = left.copy().reset_index(names='_left_row')
    right_keep = right[['RID', right_date] + fields].dropna(subset=['RID', right_date]).copy()
    candidates = base[['_left_row', 'RID', left_date]].merge(right_keep, on='RID', how='left')
    candidates['_days'] = (candidates[right_date] - candidates[left_date]).abs().dt.days
    candidates = candidates[candidates['_days'].le(max_days)].copy()
    if candidates.empty:
        for field in fields:
            base[field] = np.nan
        return base.drop(columns='_left_row')
    nearest = (candidates.sort_values(['_left_row', '_days', right_date])
                         .drop_duplicates('_left_row', keep='first')
                         [['_left_row'] + fields])
    return base.merge(nearest, on='_left_row', how='left').drop(columns='_left_row')

# PHC MRI: date, age, sex, diagnosis are already harmonized in one table.
phc = standardize_rid(to_datetime_col(phc, 'PHC_SCANDATE'))
phc['AGE'] = pd.to_numeric(phc['PHC_Age_T1'], errors='coerce')
phc['diagnosis_phc'] = phc['PHC_Diagnosis'].map(diagnosis_label)
phc['sex'] = phc['PHC_Sex'].map({1: 'Male', 2: 'Female'}).fillna(phc['PHC_Sex'].astype('string'))

index_mri = (phc.dropna(subset=['RID', 'PHC_SCANDATE', 'AGE'])
                .sort_values(['RID', 'PHC_SCANDATE'])
                .drop_duplicates('RID', keep='first')
                .rename(columns={'PHC_SCANDATE': 'MRI_DATE'})
                .copy())

# Education is a stable demographic covariate: one RID-level record is sufficient.
demo = standardize_rid(to_datetime_col(demo, 'VISDATE'))
demo['education'] = pd.to_numeric(demo['PTEDUCAT'], errors='coerce')
demo_unique = (demo.dropna(subset=['RID'])
                 .sort_values(['RID', 'VISDATE'])
                 .drop_duplicates('RID', keep='last')[['RID', 'education']])
cohort = index_mri.merge(demo_unique, on='RID', how='left', validate='one_to_one')

# Use DXSUM only as a fallback for missing PHC diagnosis.
dx = standardize_rid(to_datetime_col(dx, 'EXAMDATE'))
dx['diagnosis_dx'] = dx['DIAGNOSIS'].map(diagnosis_label)
cohort = nearest_by_rid(cohort, dx, 'MRI_DATE', 'EXAMDATE', ['diagnosis_dx'])
cohort['diagnosis'] = cohort['diagnosis_phc'].fillna(cohort['diagnosis_dx'])
cohort['eTIV'] = pd.to_numeric(cohort['EstimatedTotalIntraCranialVol_combat'], errors='coerce')

# Pre-specified eligible age range; retain only the three primary clinical groups.
cohort = cohort[cohort['AGE'].between(55, 95) & cohort['diagnosis'].isin(['CN', 'MCI', 'Dementia'])].copy()
cohort['diagnosis'] = pd.Categorical(cohort['diagnosis'], ['CN', 'MCI', 'Dementia'])

assert cohort['RID'].is_unique, 'index MRI cohort must contain one row per RID'
assert cohort['AGE'].notna().all(), 'AGE should be complete after eligibility filtering'
print(f'Index MRI records: {len(index_mri):,}')
print(f'Analysis cohort: {len(cohort):,} participants')
display(cohort['diagnosis'].value_counts().rename_axis('diagnosis').to_frame('n'))

## 3. ROI 정의와 전처리 계획

PHC 파일의 `_combat` 열은 ComBat 조화가 적용된 MRI 측정값입니다. 이 노트북에서는 해부학적 ROI를 모델에 넣되, eTIV 자체와 이미 eTIV 비율인 전역 지표는 모델 입력에서 제외합니다.

부피 ROI를 다시 단순 ICV 비율로 바꾸지 않는 이유는, 제공된 값이 이미 ComBat 조화된 측정값이기 때문입니다. eTIV는 진단군 ANCOVA의 공변량으로 사용합니다.

결측률 기준은 **CN train에서만** 계산합니다. test·MCI·Dementia의 정보를 ROI 선택에 쓰지 않기 위해서입니다.

In [ ]:
combat_columns = [c for c in phc.columns if c.endswith('_combat')]
exclude_from_model = {
    'EstimatedTotalIntraCranialVol_combat',
    'BrainSegVol.to.eTIV_combat',
    'MaskVol.to.eTIV_combat',
}
roi_candidates = [c for c in combat_columns if c not in exclude_from_model and c in cohort.columns]

# Dictionary is used only for readable anatomical labels; feature choice stays code-based and reproducible.
feature_labels = (phc_dict[['FLDNAME', 'TEXT']].dropna().drop_duplicates('FLDNAME')
                  .set_index('FLDNAME')['TEXT'].to_dict())

assert len(roi_candidates) > 100, 'Unexpectedly few PHC ComBat ROI candidates'
print(f'ComBat candidates: {len(combat_columns)}')
print(f'ROI candidates after eTIV exclusions: {len(roi_candidates)}')
print(f'Anatomical labels available: {sum(c in feature_labels for c in roi_candidates)}')

## 4. EDA: 분석 전 반드시 확인할 것

EDA(Exploratory Data Analysis)는 모델을 잘 보이게 만드는 단계가 아니라, 데이터가 연구 질문에 맞는지 확인하는 단계입니다.

여기서는 다음을 확인합니다.

- 진단군마다 표본 수와 나이 분포가 크게 다른지
- 성별, 교육연수, eTIV의 결측치가 어느 정도인지
- MRI ROI 결측률이 특정 변수에 집중되는지

진단군 간 나이 분포가 다르면 BAG도 나이 영향을 받을 수 있으므로, 이후 ANCOVA에서 나이를 보정합니다.

In [ ]:
table1 = (cohort.groupby('diagnosis', observed=False)
          .agg(n=('RID', 'size'),
               age_mean=('AGE', 'mean'), age_sd=('AGE', 'std'),
               education_mean=('education', 'mean'),
               etiv_mean=('eTIV', 'mean'))
          .round(2))
sex_counts = pd.crosstab(cohort['diagnosis'], cohort['sex'], dropna=False)
display(table1)
display(sex_counts)
table1.to_csv(TABLE_DIR / 'table1_demographics.csv')
sex_counts.to_csv(TABLE_DIR / 'table1_sex_counts.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=cohort, x='AGE', hue='diagnosis', stat='density', common_norm=False, kde=True, ax=axes[0])
axes[0].set_title('Age distribution by diagnosis')
sns.boxplot(data=cohort, x='diagnosis', y='AGE', ax=axes[1], palette='Set2')
axes[1].set_title('Age by diagnosis')
plt.tight_layout(); plt.savefig(FIG_DIR / 'eda_age_by_diagnosis.png', dpi=300); plt.show()

audit_columns = ['AGE', 'sex', 'education', 'eTIV'] + roi_candidates
missingness = cohort[audit_columns].isna().mean().sort_values(ascending=False)
display(missingness.head(20).rename('missing_fraction').to_frame())
missingness.rename('missing_fraction').to_csv(TABLE_DIR / 'missingness_all_candidates.csv')

plt.figure(figsize=(8, 5))
sns.histplot(missingness[roi_candidates], bins=25)
plt.xlabel('ROI missing fraction'); plt.title('PHC ROI missingness distribution')
plt.tight_layout(); plt.savefig(FIG_DIR / 'eda_roi_missingness.png', dpi=300); plt.show()

## 5. CN-only split과 train 기반 ROI 선택

뇌 나이 모델의 정상 기준은 CN에서만 만듭니다. 같은 RID가 여러 분할에 섞이지 않도록 `GroupShuffleSplit`에 RID를 전달합니다. 현재 index MRI는 RID당 한 행이지만, 이 원칙을 코드에 명시하면 이후 종단 분석으로 확장할 때도 안전합니다.

ROI 결측률 필터도 train CN에서만 계산합니다. 이 원칙은 test와 환자군의 분포가 학습 설계에 미리 영향을 주는 것을 막습니다.

In [ ]:
cn = cohort[cohort['diagnosis'].eq('CN')].copy()
assert cn['RID'].nunique() >= 30, 'CN sample is too small for the planned split'

split_1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED)
train_idx, hold_idx = next(split_1.split(cn, groups=cn['RID']))
cn_train, cn_hold = cn.iloc[train_idx].copy(), cn.iloc[hold_idx].copy()

split_2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=SEED + 1)
val_idx, test_idx = next(split_2.split(cn_hold, groups=cn_hold['RID']))
cn_val, cn_test = cn_hold.iloc[val_idx].copy(), cn_hold.iloc[test_idx].copy()

for name, frame in {'train': cn_train, 'validation': cn_val, 'test': cn_test}.items():
    print(f'{name:11s}: {len(frame):4,} RID | age {frame.AGE.mean():.1f} ± {frame.AGE.std():.1f}')

train_missing = cn_train[roi_candidates].isna().mean()
roi_features = train_missing[train_missing <= (1 - MIN_FEATURE_COMPLETENESS)].index.tolist()
assert len(roi_features) > 100, 'Too few ROI features after train-only completeness filter'

feature_manifest = pd.DataFrame({
    'feature_code': roi_features,
    'anatomical_label': [feature_labels.get(c, c) for c in roi_features],
    'train_missing_fraction': train_missing[roi_features].values,
})
feature_manifest.to_csv(TABLE_DIR / 'roi_feature_manifest.csv', index=False)
print(f'Final MRI ROI features: {len(roi_features)}')
display(feature_manifest.head())

## 6. 모델 설계와 학습

모델은 MRI ROI와 성별로 실제 나이를 예측합니다.

- `DummyRegressor`: 평균 나이만 예측하는 최소 기준선
- `Ridge`: 상관된 ROI를 안정적으로 다루는 사전 지정 주 모델
- `Extra Trees`: 비선형 관계가 도움이 되는지 확인하는 비교 모델

중앙값 대치·표준화·one-hot 인코딩은 모두 `Pipeline` 안에 들어 있습니다. 따라서 각 CV fold의 train 부분에서만 학습되어 누수를 막습니다. 결과가 조금 더 좋아도, Ridge와 비선형 모델의 성능 차이가 작다면 해석 가능성이 높은 Ridge를 주 모델로 유지합니다.

In [ ]:
model_features = roi_features + ['sex']

preprocessor = ColumnTransformer([
    ('roi', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), roi_features),
    ('sex', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), ['sex']),
])

model_specs = {
    'Dummy mean age': (
        Pipeline([('prep', preprocessor), ('model', DummyRegressor())]), {}
    ),
    'Ridge': (
        Pipeline([('prep', preprocessor), ('model', Ridge())]),
        {'model__alpha': [0.1, 1.0, 10.0, 100.0]}
    ),
    'Extra Trees': (
        Pipeline([('prep', preprocessor), ('model', ExtraTreesRegressor(random_state=SEED, n_jobs=N_JOBS))]),
        {'model__n_estimators': [300], 'model__min_samples_leaf': [1, 3]}
    ),
}

cv = GroupKFold(n_splits=5)
searches, cv_rows = {}, []
for name, (pipeline, params) in model_specs.items():
    search = GridSearchCV(
        estimator=pipeline, param_grid=params, scoring='neg_mean_absolute_error',
        cv=cv, n_jobs=N_JOBS, refit=True
    )
    search.fit(cn_train[model_features], cn_train['AGE'], groups=cn_train['RID'])
    searches[name] = search
    cv_rows.append({
        'model': name,
        'cv_mae': -search.best_score_,
        'best_parameters': str(search.best_params_),
    })

cv_results = pd.DataFrame(cv_rows).sort_values('cv_mae')
display(cv_results.round(3))
cv_results.to_csv(TABLE_DIR / 'model_selection_cv.csv', index=False)

# Pre-specified primary model: Ridge. Extra Trees remains a transparent comparator.
age_model = searches['Ridge'].best_estimator_
print('Primary Brain Age model: Ridge')

## 7. Validation 기반 나이 편향 보정

뇌 나이 모델은 젊은 사람은 다소 늙게, 고령자는 다소 젊게 예측하는 경향이 있습니다. 이 경향을 보정하지 않으면 BAG가 실제 나이와 연결되어 질병 효과처럼 보일 수 있습니다.

validation CN에서만 아래 관계를 적합하고 계수를 고정합니다.

```text
raw predicted age = α + β × chronological age
corrected predicted age = (raw predicted age − α) / β
BAG = corrected predicted age − chronological age
```

test와 MCI·Dementia는 이 보정식을 새로 학습하는 데 사용하지 않습니다.

In [ ]:
val_raw_prediction = age_model.predict(cn_val[model_features])
bias_model = LinearRegression().fit(cn_val[['AGE']], val_raw_prediction)
alpha = float(bias_model.intercept_)
beta = float(bias_model.coef_[0])
assert beta > 0.05, f'Unexpected bias slope: {beta:.3f}'

bias_parameters = pd.DataFrame([{'alpha': alpha, 'beta': beta, 'validation_n': len(cn_val)}])
bias_parameters.to_csv(TABLE_DIR / 'bag_bias_correction_parameters.csv', index=False)
print(f'Frozen bias correction: raw_predicted_age = {alpha:.3f} + {beta:.3f} × actual_age')

def score_bag(frame):
    out = frame.copy()
    out['predicted_age_raw'] = age_model.predict(out[model_features])
    out['predicted_age_corrected'] = (out['predicted_age_raw'] - alpha) / beta
    out['BAG'] = out['predicted_age_corrected'] - out['AGE']
    return out

## 8. 독립 CN test 평가

이 셀은 한 번도 모델 학습·ROI 선택·편향 보정에 쓰지 않은 CN test를 평가합니다.

- MAE: 평균 예측 오차(년)
- RMSE: 큰 오차에 더 민감한 지표
- R²: 실제 나이 차이를 설명하는 정도
- BAG–Age correlation: 보정 후 BAG가 실제 나이와 남아 있는 관계
- Dummy 대비 개선: MRI 입력이 평균 나이 예측보다 유용한지

test 결과를 본 뒤 모델·ROI·분할을 다시 바꾸면 test는 더 이상 독립 검증이 아닙니다.

In [ ]:
cn_test_scored = score_bag(cn_test)
dummy_prediction = searches['Dummy mean age'].best_estimator_.predict(cn_test[model_features])

test_performance = pd.DataFrame([
    {
        'model': 'Ridge (raw)',
        'MAE': mean_absolute_error(cn_test['AGE'], cn_test_scored['predicted_age_raw']),
        'RMSE': mean_squared_error(cn_test['AGE'], cn_test_scored['predicted_age_raw']) ** 0.5,
        'R2': r2_score(cn_test['AGE'], cn_test_scored['predicted_age_raw']),
        'BAG_age_r': (cn_test_scored['predicted_age_raw'] - cn_test_scored['AGE']).corr(cn_test_scored['AGE']),
    },
    {
        'model': 'Ridge (bias-corrected)',
        'MAE': mean_absolute_error(cn_test['AGE'], cn_test_scored['predicted_age_corrected']),
        'RMSE': mean_squared_error(cn_test['AGE'], cn_test_scored['predicted_age_corrected']) ** 0.5,
        'R2': r2_score(cn_test['AGE'], cn_test_scored['predicted_age_corrected']),
        'BAG_age_r': cn_test_scored['BAG'].corr(cn_test_scored['AGE']),
    },
    {
        'model': 'Dummy mean age',
        'MAE': mean_absolute_error(cn_test['AGE'], dummy_prediction),
        'RMSE': mean_squared_error(cn_test['AGE'], dummy_prediction) ** 0.5,
        'R2': r2_score(cn_test['AGE'], dummy_prediction),
        'BAG_age_r': np.nan,
    },
])
test_performance['MAE_improvement_vs_dummy'] = test_performance.loc[2, 'MAE'] - test_performance['MAE']
display(test_performance.round(3))
test_performance.to_csv(TABLE_DIR / 'model_performance_heldout_cn_test.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.scatterplot(data=cn_test_scored, x='AGE', y='predicted_age_corrected', ax=axes[0])
lims = [min(cn_test_scored['AGE'].min(), cn_test_scored['predicted_age_corrected'].min()),
        max(cn_test_scored['AGE'].max(), cn_test_scored['predicted_age_corrected'].max())]
axes[0].plot(lims, lims, '--', color='black', label='perfect prediction')
axes[0].set(xlabel='Actual age', ylabel='Bias-corrected predicted age', title='Held-out CN test')
axes[0].legend()
sns.regplot(data=cn_test_scored, x='AGE', y='BAG', ax=axes[1], scatter_kws={'alpha': 0.7})
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Held-out CN: BAG vs actual age')
plt.tight_layout(); plt.savefig(FIG_DIR / 'heldout_cn_test_evaluation.png', dpi=300); plt.show()

## 9. 진단군 BAG 비교: 사전 지정 ANCOVA

동결된 Ridge 모델과 동결된 bias correction을 전체 코호트에 적용합니다. 이후 ANCOVA로 다음을 평가합니다.

```text
BAG ~ diagnosis + age + sex + education + eTIV
```

이는 단순히 진단군 평균을 비교하는 것이 아니라, 나이·성별·교육·두개강 용적 차이를 고려한 뒤에도 진단군의 BAG가 다른지 평가하는 방법입니다. 이 모형은 모든 진단군에서 BAG와 나이의 관계 기울기가 같다고 가정하므로, 다음 단계에서 그 가정을 직접 점검합니다.

In [ ]:
scored = score_bag(cohort)
scored['diagnosis'] = pd.Categorical(scored['diagnosis'], ['CN', 'MCI', 'Dementia'])
scored.to_csv(PROCESSED_DIR / 'phc_index_mri_scored_bag.csv', index=False)

bag_summary = (scored.groupby('diagnosis', observed=False)['BAG']
               .agg(n='count', mean='mean', sd='std', median='median')
               .round(3))
display(bag_summary)
bag_summary.to_csv(TABLE_DIR / 'bag_summary_by_diagnosis.csv')

plt.figure(figsize=(8, 5))
sns.boxplot(data=scored, x='diagnosis', y='BAG', palette='Set2', showfliers=False)
sns.stripplot(data=scored, x='diagnosis', y='BAG', color='0.25', alpha=0.35, size=3)
plt.axhline(0, color='black', lw=1)
plt.ylabel('Bias-corrected BAG (years)')
plt.title('Brain Age Gap across clinical diagnosis groups')
plt.tight_layout(); plt.savefig(FIG_DIR / 'bag_by_diagnosis.png', dpi=300); plt.show()

primary_df = scored.dropna(subset=['BAG', 'AGE', 'sex', 'education', 'eTIV', 'diagnosis']).copy()
primary_fit = smf.ols('BAG ~ C(diagnosis) + AGE + C(sex) + education + eTIV', data=primary_df).fit()
primary_ancova = anova_lm(primary_fit, typ=2, robust='hc3')
display(primary_ancova)
primary_ancova.to_csv(TABLE_DIR / 'ancova_diagnosis_hc3.csv')

# Robust coefficient table: useful for direction and confidence intervals.
robust_fit = primary_fit.get_robustcov_results(cov_type='HC3')
coef_table = pd.DataFrame({
    'term': robust_fit.model.exog_names,
    'estimate': robust_fit.params,
    'std_error_HC3': robust_fit.bse,
    'p_value': robust_fit.pvalues,
    'ci_low': robust_fit.conf_int()[:, 0],
    'ci_high': robust_fit.conf_int()[:, 1],
})
display(coef_table.round(4))
coef_table.to_csv(TABLE_DIR / 'diagnosis_model_coefficients_hc3.csv', index=False)
print(f'Primary ANCOVA sample: {len(primary_df):,}')

## 10. 필수 점검: 진단군에 따라 BAG–나이 관계가 다른가?

앞선 ANCOVA는 모든 진단군에서 BAG와 실제 나이의 관계 기울기가 같다고 가정합니다. 이 가정이 맞지 않으면 하나의 평균 진단군 차이로 결론을 내리면 안 됩니다. 먼저 진단군별 BAG–나이 상관을 확인하고, 그 다음 `diagnosis × AGE` 상호작용을 검정합니다.

상호작용이 유의하면 진단군 차이는 특정 기준 나이에서 CN 대비 차이로 보고합니다. 이 분석은 사후적으로 유리한 결과를 찾기 위한 것이 아니라, BAG가 나이에 덜 의존해야 한다는 연구 설계의 핵심 가정을 점검하는 단계입니다.

In [ ]:
# Diagnostic check: residual BAG--age association within each clinical group.
age_correlation_rows = []
for diagnosis, group in primary_df.groupby('diagnosis', observed=False):
    group = group.dropna(subset=['BAG', 'AGE'])
    if len(group) >= 3 and group['AGE'].nunique() > 1:
        r_value, p_value = pearsonr(group['BAG'], group['AGE'])
    else:
        r_value, p_value = np.nan, np.nan
    age_correlation_rows.append({'diagnosis': diagnosis, 'n': len(group),
                                 'BAG_AGE_r': r_value, 'p_value': p_value})

bag_age_by_diagnosis = pd.DataFrame(age_correlation_rows)
display(bag_age_by_diagnosis.round(4))
bag_age_by_diagnosis.to_csv(TABLE_DIR / 'bag_age_correlation_by_diagnosis.csv', index=False)

interaction_fit = smf.ols(
    'BAG ~ C(diagnosis) * AGE + C(sex) + education + eTIV', data=primary_df
).fit()
interaction_ancova = anova_lm(interaction_fit, typ=2, robust='hc3')
display(interaction_ancova)
interaction_ancova.to_csv(TABLE_DIR / 'ancova_diagnosis_by_age_interaction_hc3.csv')

interaction_p = interaction_ancova.loc['C(diagnosis):AGE', 'PR(>F)']
print(f'Diagnosis × AGE interaction p-value: {interaction_p:.3e}')
if interaction_p < 0.05:
    print('Use age-specific CN contrasts below as the primary group comparison.')
else:
    print('No strong evidence of different age slopes; the simpler ANCOVA is adequate.')

## 11. 최종 주 결과: 기준 나이별 CN 대비 BAG 차이

상호작용 모형에서 65세·75세·85세를 기준으로 MCI와 Dementia의 조정 BAG를 CN과 비교합니다. 성별은 Female(기준 범주), 교육연수와 eTIV는 분석 표본의 평균으로 고정합니다. 이 고정값은 대비의 공정한 기준을 만들기 위한 것이며, 핵심은 같은 나이와 같은 공변량 조건에서 진단군 간 BAG가 얼마나 다른지입니다.

HC3 robust standard error와 95% 신뢰구간을 사용합니다. 상호작용이 유의하다면 이 표가 보고서와 발표에서 사용할 최종 진단군 결과표입니다.

In [ ]:
robust_interaction = interaction_fit.get_robustcov_results(cov_type='HC3')
term_index = {term: i for i, term in enumerate(robust_interaction.model.exog_names)}

def cn_contrast_at_age(diagnosis, reference_age):
    """Robust MCI/CN or Dementia/CN BAG contrast at a specified age."""
    main_term = f'C(diagnosis)[T.{diagnosis}]'
    interaction_term = f'C(diagnosis)[T.{diagnosis}]:AGE'
    missing_terms = [term for term in [main_term, interaction_term] if term not in term_index]
    if missing_terms:
        raise KeyError(f'Expected diagnosis terms are missing: {missing_terms}')
    contrast = np.zeros(len(term_index))
    contrast[term_index[main_term]] = 1.0
    contrast[term_index[interaction_term]] = float(reference_age)
    result = robust_interaction.t_test(contrast)
    ci_low, ci_high = np.asarray(result.conf_int(alpha=0.05)).ravel()
    return {
        'reference_age': reference_age,
        'contrast': f'{diagnosis} vs CN',
        'BAG_difference_years': float(np.asarray(result.effect).squeeze()),
        'std_error_HC3': float(np.asarray(result.sd).squeeze()),
        'p_value': float(np.asarray(result.pvalue).squeeze()),
        'ci_low': float(ci_low),
        'ci_high': float(ci_high),
    }

contrast_rows = [
    cn_contrast_at_age(diagnosis, age)
    for age in [65, 75, 85]
    for diagnosis in ['MCI', 'Dementia']
]
age_specific_contrasts = pd.DataFrame(contrast_rows)
display(age_specific_contrasts.round(4))
age_specific_contrasts.to_csv(TABLE_DIR / 'age_specific_bag_contrasts_vs_cn_hc3.csv', index=False)

# Visualize adjusted diagnosis trajectories. This is descriptive; the table above is the inferential result.
age_grid = np.linspace(primary_df['AGE'].quantile(0.02), primary_df['AGE'].quantile(0.98), 100)
reference_sex = 'Female' if 'Female' in primary_df['sex'].dropna().unique() else primary_df['sex'].mode().iat[0]
prediction_frames = []
for diagnosis in ['CN', 'MCI', 'Dementia']:
    new_data = pd.DataFrame({
        'diagnosis': diagnosis, 'AGE': age_grid, 'sex': reference_sex,
        'education': primary_df['education'].mean(), 'eTIV': primary_df['eTIV'].mean(),
    })
    predicted = interaction_fit.get_prediction(new_data).summary_frame(alpha=0.05)
    new_data['predicted_BAG'] = predicted['mean'].to_numpy()
    prediction_frames.append(new_data)

adjusted_trajectories = pd.concat(prediction_frames, ignore_index=True)
plt.figure(figsize=(8, 5))
sns.lineplot(data=adjusted_trajectories, x='AGE', y='predicted_BAG', hue='diagnosis', linewidth=2.5)
plt.axhline(0, color='black', lw=1)
plt.ylabel('Adjusted predicted BAG (years)')
plt.title('Diagnosis-specific BAG trajectories from interaction model')
plt.tight_layout(); plt.savefig(FIG_DIR / 'bag_diagnosis_age_interaction.png', dpi=300); plt.show()

## 12. 생물학적 의미: ROI 중요도

Permutation importance는 held-out CN test에서 한 ROI의 값을 무작위로 섞었을 때 MAE가 얼마나 나빠지는지 계산합니다. MAE 증가가 큰 ROI일수록 뇌 나이 예측에 유용했던 특징입니다.

이 결과는 **예측 기여도**입니다. 특정 ROI가 중요하다는 사실만으로 그 부위가 알츠하이머병의 원인이라는 결론을 내릴 수는 없습니다. 결과를 해마·내후각피질·측두엽·뇌실 등 알려진 노화 관련 구조와 연결해 조심스럽게 설명합니다.

In [ ]:
perm = permutation_importance(
    age_model, cn_test[model_features], cn_test['AGE'],
    scoring='neg_mean_absolute_error', n_repeats=20, random_state=SEED, n_jobs=N_JOBS
)
importance = pd.DataFrame({
    'feature_code': model_features,
    'mae_increase_when_permuted': perm.importances_mean,
    'importance_sd': perm.importances_std,
})
importance['anatomical_label'] = importance['feature_code'].map(feature_labels).fillna(importance['feature_code'])
importance = importance.sort_values('mae_increase_when_permuted', ascending=False)
display(importance.head(15))
importance.to_csv(TABLE_DIR / 'permutation_importance_heldout_cn_test.csv', index=False)

top = importance.head(15).sort_values('mae_increase_when_permuted')
plt.figure(figsize=(9, 7))
plt.barh(top['anatomical_label'], top['mae_increase_when_permuted'])
plt.xlabel('Increase in held-out CN MAE (years)')
plt.title('Permutation importance: contribution to brain-age prediction')
plt.tight_layout(); plt.savefig(FIG_DIR / 'permutation_importance_top15.png', dpi=300); plt.show()

## 13. 결과를 말로 해석하는 틀

실행 후 아래 문장 틀에 실제 결과를 넣어 발표·README를 작성하세요.

1. **모델**: “CN 독립 test에서 Ridge의 보정 전/후 MAE는 각각 __년이었고, Dummy 기준선보다 __년 개선되었다.”
2. **보정 확인**: “보정 후 CN test BAG와 실제 나이의 상관은 __로, 나이 의존성이 얼마나 줄었는지 확인했다.”
3. **주 결과**: “진단군 × 나이 상호작용은 __였으므로, 75세에서 MCI와 Dementia의 조정 BAG는 CN보다 각각 __년과 __년 높았다.”
4. **생물학적 의미**: “나이 예측에 기여한 상위 ROI는 __였으며, 이는 정상 노화와 관련된 구조적 차이를 반영할 가능성이 있다. 단, 인과관계는 주장하지 않는다.”

### 다음 확장 단계

MVP의 모든 표·그림·해석이 완성된 뒤에만 다음을 진행합니다.

- Amyloid PET를 MRI와 ±180일 이내로 연결해 병리 부차 분석
- CN train 기반 normative ROI Z-score로 해부학적 aging burden 분석
- ROI 집합·eTIV 처리·모델을 바꾼 민감도 분석
- **뇌 표면 시각화 확장**: 분석 정의를 고정한 뒤 Desikan-Killiany 피질 두께 parcel 68개의 BAG 연관성을 HC3 회귀와 FDR 보정으로 평가하고, fsaverage5 표면에 정적 PNG·회전형 HTML로 표시
- `src/` 함수화, README 정리, 결과 자동 저장으로 GitHub 포트폴리오 완성

원본 ADNI 데이터와 참가자 수준 결과는 데이터 사용 계약에 따라 공개 저장소에 올리지 않습니다.